# GDP Analysis — Part 4: Comparing GDP Growth Across Countries (Advanced)

This notebook builds on the `GDP_Growth` column computed in Part 1 to:

1. Generate an individual GDP-growth chart for every country in bulk
2. Build a **reusable function** to compare growth rates between any set of countries on demand
3. Handle inconsistent year coverage across countries when comparing many of them at once

Uses `data/gdp_with_growth.csv`, produced by `01_dataset_walkthrough_and_gdp_growth.ipynb` — run that notebook first if this file doesn't exist yet.

In [1]:
import os
import pandas as pd
import plotly.express as px
import plotly.offline as pyo

df = pd.read_csv('data/gdp_with_growth.csv')
df.head()

,Country Name,Country Code,Year,Value,GDP_Growth
0,Arab World,ARB,1968,2.576068e+10,0.00
1,Arab World,ARB,1969,2.843420e+10,10.38
2,Arab World,ARB,1970,3.138550e+10,10.38
3,Arab World,ARB,1971,3.642691e+10,16.06
4,Arab World,ARB,1972,4.331606e+10,18.91


## 1. GDP Growth for Every Country, in Bulk

In [2]:
os.makedirs('output/individual_gdp_growth', exist_ok=True)

for country_code in df['Country Code'].unique():
    country_df = df[df['Country Code'] == country_code]
    fig = px.line(country_df, x='Year', y='GDP_Growth', title=country_code + ' GDP Growth')
    pyo.plot(fig, filename=f'output/individual_gdp_growth/{country_code}.html', auto_open=False)

print('Generated', df['Country Code'].nunique(), 'individual GDP growth charts.')

Generated 256 individual GDP growth charts.


## 2. A Reusable Comparison Function

Rather than writing the same plotting code every time a new comparison is needed, this wraps it in a function that takes any list of country codes and produces one combined chart.

In [3]:
def compare_gdp_growth(country_codes, auto_open=False):
    """Plot GDP growth (%) for the given list of country codes on a single chart.

    Saves the result as an HTML file named after the countries compared, e.g. 'IND_USA.html'.
    """
    frames = [df[df['Country Code'] == code] for code in country_codes]
    compare_df = pd.concat(frames, axis=0)

    fig = px.line(
        compare_df, x='Year', y='GDP_Growth',
        title='GDP Growth Comparison | ' + ' vs. '.join(country_codes),
        color='Country Name'
    )
    os.makedirs('output', exist_ok=True)
    pyo.plot(fig, filename='output/' + '_'.join(country_codes) + '_growth.html', auto_open=auto_open)
    return fig

In [4]:
compare_gdp_growth(['IND', 'USA', 'ITA', 'CHN'])

## 3. Comparing Every Country's Growth at Once — Handling Missing Data

Not every country has the same year range (some have as few as 2 years recorded). Plotting all 256 at once would be misleading — countries with sparse data would appear to have wildly erratic growth. To keep the comparison fair, this filters down to only the countries with the **full 1960–2016 range** (57 years) before comparing.

In [5]:
complete_frames = [
    df[df['Country Name'] == name]
    for name in df['Country Name'].unique()
    if len(df[df['Country Name'] == name]) == 57
]

df_complete = pd.concat(complete_frames, axis=0)
print(f"{df_complete['Country Name'].nunique()} of {df['Country Name'].nunique()} countries have the full 1960-2016 range.")

120 of 256 countries have the full 1960-2016 range.


In [6]:
fig = px.line(df_complete, x='Year', y='GDP_Growth', title='GDP Growth (1960–2016) — Countries with Complete Data', color='Country Name')
pyo.plot(fig, filename='output/gdp_growth_1960_2016_complete.html', auto_open=False)

'output/gdp_growth_1960_2016_complete.html'

**Observation:** Filtering to countries with complete year coverage avoids the misleading spikes that show up when a country only has 2–3 years of data — a small base makes any single year's growth number look far more extreme than it actually is.